# 05. PyTorch 핵심: Tensor · Autograd · Device · 디버깅

이 노트북의 목표는 API 암기가 아니라 **텐서가 모델 안에서 어떻게 이동하고 미분되는지 추적하는 습관**을 만드는 것입니다.

예상 시간: 60~90분 · 실행 장치: CUDA → MPS → CPU 자동 선택 · 선행 지식: Python 함수/클래스, NumPy 기초

> 권장 방식: 각 셀을 그대로 한 번 실행한 뒤, 숫자·shape·축을 바꾸어 다시 실행하세요. 오류 메시지를 읽는 과정도 실습입니다.

## 이번 노트북에서 익힐 것

- `shape`, `dtype`, `device`를 데이터 계약(data contract)으로 읽기
- broadcasting과 차원 변환을 작은 예제로 확인하기
- autograd 그래프, leaf tensor, gradient 누적을 이해하기
- `no_grad`, `inference_mode`, `detach`의 역할 구분하기
- NaN/Inf, 잘못된 shape, 끊어진 gradient를 빠르게 진단하기
- 작은 최적화 루프를 직접 작성하고 수치 미분으로 검산하기

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.manual_seed,sys.path.insert -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.manual_seed(seed)</code></summary>

#### `torch.manual_seed(seed)`

- **역할:** PyTorch 난수 생성기의 시드를 설정합니다.
- **입력·반환:** 정수 시드를 받고 `Generator`를 반환합니다.
- **이 셀에서 쓰는 이유:** 가중치 초기화와 텐서 샘플링을 반복 가능하게 만듭니다.
- **주의점:** 완전한 결정성에는 알고리즘·장치 설정도 추가로 필요할 수 있습니다.
- **공식 문서:** [torch.manual_seed](https://docs.pytorch.org/docs/stable/generated/torch.manual_seed.html)

</details>

<details>
<summary><code>sys.path.insert(index, path)</code></summary>

#### `sys.path.insert(index, path)`

- **역할:** Python 모듈 검색 경로의 지정 위치에 경로를 추가합니다.
- **입력·반환:** index와 경로 문자열을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 노트북에서 프로젝트 `src` 모듈을 import할 수 있게 합니다.
- **주의점:** 전역 import 상태를 바꾸므로 중복·우선순위와 실행 위치에 주의합니다.
- **공식 문서:** [sys.path.insert](https://docs.python.org/3/library/sys.html#sys.path)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
from pathlib import Path
import sys
import torch
from torch import nn

torch.manual_seed(42)
torch.set_printoptions(precision=4, sci_mode=False)

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "pyproject.toml").exists()), None)
assert PROJECT_ROOT is not None, "프로젝트 폴더 안에서 노트북을 열어 주세요."
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
from llm_engineering_lab.acceleration import get_accelerator

ACCELERATOR = get_accelerator()

print("project:", PROJECT_ROOT)
print("torch:", torch.__version__)
print(ACCELERATOR.summary())

## 1. Tensor는 값 + 메타데이터다

텐서를 볼 때는 값만 보지 말고 다음 네 가지를 함께 확인합니다.

1. `shape`: 각 축의 의미는 무엇인가?
2. `dtype`: 정수 ID인가, 실수 특징인가?
3. `device`: CPU와 GPU 중 어디에 있는가?
4. `requires_grad`: 이 값까지 gradient를 계산해야 하는가?

LLM에서 흔한 표기는 `[B, T, D] = [batch, time, hidden]`입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.tensor,torch.randn,torch.float32 -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.tensor(data, dtype=None, device=None)</code></summary>

#### `torch.tensor(data, dtype=None, device=None)`

- **역할:** 데이터를 복사해 새 PyTorch 텐서를 만듭니다.
- **입력·반환:** 배열형 데이터와 선택적 dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 상수·label·축소 실험 입력을 명시적으로 구성합니다.
- **주의점:** 기존 텐서에서 호출하면 복사가 생기므로 경우에 따라 `as_tensor`가 낫습니다.
- **공식 문서:** [torch.tensor](https://docs.pytorch.org/docs/stable/generated/torch.tensor.html)

</details>

<details>
<summary><code>torch.randn(*size, generator=None, ...)</code></summary>

#### `torch.randn(*size, generator=None, ...)`

- **역할:** 표준정규분포에서 난수 텐서를 샘플링합니다.
- **입력·반환:** shape와 선택적 생성기를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 합성 입력·noise·초기 실험값을 만듭니다.
- **주의점:** 재현하려면 해당 PyTorch 생성기의 시드를 먼저 고정해야 합니다.

##### 관련 수식과 코드 연결

$$
X\sim\mathcal N(0,1),\qquad f(x)=\frac{1}{\sqrt{2\pi}}\exp\!\left(-\frac{x^2}{2}\right)
$$

- **기호:** $\mathcal N(0,1)$은 평균 0, 분산 1인 표준정규분포입니다.
- **수식 → 코드:** 요청한 모든 위치에서 독립적인 표준정규 난수를 샘플링합니다.
- **직관:** 학습 noise와 합성 입력의 분포 가정을 코드로 만듭니다.
- **shape 확인:** 인자로 지정한 shape 그대로 반환하며 generator·device가 재현성에 영향을 줍니다.
- **공식 문서:** [torch.randn](https://docs.pytorch.org/docs/stable/generated/torch.randn.html)

</details>

<details>
<summary><code>torch.float32</code></summary>

#### `torch.float32`

- **역할:** 32비트 부동소수점 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** 일반적인 학습 실수 dtype입니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.float32](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
token_ids = torch.tensor([[2, 10, 11, 3], [2, 20, 3, 0]], dtype=torch.long)
embeddings = torch.randn(2, 4, 8, dtype=torch.float32)

for name, tensor in {"token_ids": token_ids, "embeddings": embeddings}.items():
    print(
        f"{name:12s} shape={tuple(tensor.shape)}, "
        f"dtype={tensor.dtype}, device={tensor.device}"
    )

assert token_ids.shape == (2, 4)  # [B, T]
assert embeddings.shape == (2, 4, 8)  # [B, T, D]
assert token_ids.dtype == torch.long

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.float,nn.Embedding -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.float()</code></summary>

#### `tensor.float()`

- **역할:** 텐서를 32비트 부동소수점 dtype으로 변환합니다.
- **입력·반환:** 텐서를 받고 `torch.float32` 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 정수 입력이나 NumPy 데이터를 신경망 weight와 계산 가능한 dtype으로 맞춥니다.
- **주의점:** class index는 보통 정수 dtype이어야 하므로 모든 텐서에 적용하면 안 됩니다.
- **공식 문서:** [Tensor.float](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.float.html)

</details>

<details>
<summary><code>nn.Embedding(num_embeddings, embedding_dim, ...)</code></summary>

#### `nn.Embedding(num_embeddings, embedding_dim, ...)`

- **역할:** 정수 ID를 학습 가능한 벡터로 조회합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
e_i=E[i],\qquad E\in\mathbb{R}^{V\times D}
$$

- **기호:** $i$는 정수 ID, $V$는 vocabulary 크기, $D$는 embedding 차원입니다.
- **수식 → 코드:** forward는 one-hot 행렬곱을 만들지 않고 weight 행 $E[i]$를 직접 조회합니다.
- **직관:** 범주 ID를 학습 가능한 연속 벡터 좌표로 바꿉니다.
- **shape 확인:** 입력 `[...] -> 출력 [..., D]`; 입력 dtype은 보통 `torch.int64`입니다.
- **공식 문서:** [nn.Embedding](https://docs.pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# dtype에 따라 가능한 연산이 달라집니다.
ids = torch.tensor([1, 2, 3])
values = torch.tensor([1.0, 2.0, 3.0])
mask = torch.tensor([True, False, True])

print(ids.dtype, values.dtype, mask.dtype)
print("정수 -> 실수:", ids.float())
print("마스크로 선택:", values[mask])

embedding = nn.Embedding(num_embeddings=10, embedding_dim=4)
embedded = embedding(ids)  # Embedding의 인덱스는 torch.long이어야 합니다.
print("embedded shape:", embedded.shape)
assert embedded.shape == (3, 4)

## 2. Shape를 바꾸는 연산

- `unsqueeze`: 길이 1인 축 추가
- `squeeze`: 길이 1인 축 제거
- `reshape`: 원소 수를 유지하며 모양 변경
- `transpose`: 두 축 교환
- `permute`: 전체 축 순서 재배열

`view`는 메모리가 contiguous인지에 영향을 받습니다. 학습 코드에서는 의도를 분명히 하기 위해 `reshape`를 먼저 쓰는 편이 안전합니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.arange,Tensor.transpose,Tensor.reshape,torch.equal -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.arange(start=0, end, step=1, ...)</code></summary>

#### `torch.arange(start=0, end, step=1, ...)`

- **역할:** 일정 간격의 1차원 텐서를 만듭니다.
- **입력·반환:** 범위와 간격을 받고 1차원 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** position·index·time step을 벡터화해 만듭니다.
- **주의점:** 부동소수점 step은 반올림 때문에 원소 수가 예상과 다를 수 있습니다.
- **공식 문서:** [torch.arange](https://docs.pytorch.org/docs/stable/generated/torch.arange.html)

</details>

<details>
<summary><code>tensor.transpose(dim0, dim1)</code></summary>

#### `tensor.transpose(dim0, dim1)`

- **역할:** 두 축의 위치를 맞바꿉니다.
- **입력·반환:** 두 축 번호를 받고 축이 교환된 view를 반환합니다.
- **이 셀에서 쓰는 이유:** attention 행렬곱 등에서 축 순서를 맞춥니다.
- **주의점:** 반환 텐서는 비연속일 수 있어 이후 `view` 전에 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
Y_{i_{\pi(1)},\ldots,i_{\pi(K)}}=X_{i_1,\ldots,i_K}
$$

- **기호:** $\pi$는 축의 새 순서를 나타내는 permutation입니다.
- **수식 → 코드:** `transpose`는 두 축만 바꾸고 `permute`는 모든 축의 새 순서를 명시합니다.
- **직관:** 원소 값을 바꾸지 않고 각 index가 의미하는 축의 위치만 다시 배치합니다.
- **shape 확인:** 예: `[B,T,H,D] -> [B,H,T,D]`; 결과는 비연속 view일 수 있습니다.
- **공식 문서:** [Tensor.transpose](https://docs.pytorch.org/docs/stable/generated/torch.transpose.html)

</details>

<details>
<summary><code>tensor.reshape(*shape)</code></summary>

#### `tensor.reshape(*shape)`

- **역할:** 원소 수를 유지하며 텐서 shape을 바꿉니다.
- **입력·반환:** 새 shape을 받고 가능하면 view, 아니면 복사된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch·head·sequence 축을 논문 수식의 shape에 맞춥니다.
- **주의점:** 원소 수가 같아야 하며 메모리 공유 여부에 의존하면 안 됩니다.

##### 관련 수식과 코드 연결

$$
\prod_{k=1}^{K} d_k=\prod_{j=1}^{J} d'_j
$$

- **기호:** $d_k$는 원래 각 축 크기, $d'_j$는 바꾼 뒤 각 축 크기입니다.
- **수식 → 코드:** `reshape`·`view`·`flatten`은 원소의 총개수를 보존하면서 축 해석만 바꿉니다.
- **직관:** 데이터 값을 계산하는 연산이 아니라 같은 원소를 다른 좌표계로 보는 연산입니다.
- **shape 확인:** 예: `[B, H, T, D] -> [B, T, H*D]`; 왼쪽과 오른쪽 원소 수가 같아야 합니다.
- **공식 문서:** [Tensor.reshape](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.reshape.html)

</details>

<details>
<summary><code>torch.equal(input, other)</code></summary>

#### `torch.equal(input, other)`

- **역할:** 두 텐서의 크기와 원소가 정확히 같은지 검사합니다.
- **입력·반환:** 두 텐서를 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 토큰 ID나 정수 마스크처럼 완전 일치가 필요한 결과를 확인합니다.
- **주의점:** dtype이 달라도 값이 같으면 참일 수 있고 NaN은 자기 자신과 같지 않습니다.
- **공식 문서:** [torch.equal](https://docs.pytorch.org/docs/stable/generated/torch.equal.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
x = torch.arange(24).reshape(2, 3, 4)  # [B=2, T=3, D=4]
print("x:              ", x.shape)
print("unsqueeze(1):   ", x.unsqueeze(1).shape)
print("transpose(1, 2):", x.transpose(1, 2).shape)
print("flatten B,T:    ", x.reshape(-1, x.size(-1)).shape)

flat = x.reshape(-1, 4)
restored = flat.reshape(2, 3, 4)
assert torch.equal(x, restored)

In [ ]:
# Broadcasting: 뒤쪽 축부터 크기가 같거나, 둘 중 하나가 1이어야 합니다.
batch = torch.arange(6, dtype=torch.float32).reshape(2, 3)  # [B, D]
bias = torch.tensor([10.0, 20.0, 30.0])  # [D]
scale = torch.tensor([[1.0], [2.0]])  # [B, 1]

shifted = batch + bias
scaled = batch * scale
print("batch + bias:\n", shifted)
print("batch * scale:\n", scaled)
assert shifted.shape == scaled.shape == (2, 3)

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.isfinite -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.isfinite(input)</code></summary>

#### `torch.isfinite(input)`

- **역할:** 텐서의 각 원소가 유한수인지 검사합니다.
- **입력·반환:** 텐서를 받고 같은 shape의 불리언 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 gradient의 NaN·무한대 발생을 빠르게 찾습니다.
- **주의점:** 검사 결과는 미분 대상이 아니며 원인 자체를 고치지는 않습니다.
- **공식 문서:** [torch.isfinite](https://docs.pytorch.org/docs/stable/generated/torch.isfinite.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
def inspect(name: str, tensor: torch.Tensor) -> None:
    print(
        f"{name}: shape={tuple(tensor.shape)}, dtype={tensor.dtype}, "
        f"device={tensor.device}, finite={torch.isfinite(tensor).all().item()}"
    )


a = torch.randn(2, 3, 4)
b = torch.randn(2, 4)
inspect("a", a)
inspect("b", b)

try:
    _ = a + b
except RuntimeError as error:
    print("예상한 shape 오류:", str(error).splitlines()[0])

# b를 [B, 1, D]로 바꾸면 time 축으로 broadcasting됩니다.
fixed = a + b.unsqueeze(1)
inspect("fixed", fixed)
assert fixed.shape == a.shape

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.mean,Tensor.std,torch.allclose,torch.zeros -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.mean(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.mean(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 산술평균을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** batch loss, feature 통계, metric을 하나의 대표값으로 축약합니다.
- **주의점:** 어느 축을 없애는지와 빈 텐서에서 NaN이 나올 수 있음을 확인합니다.

##### 관련 수식과 코드 연결

$$
\bar x=\frac{1}{N}\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택한 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`가 평균에 포함할 원소와 결과에서 사라질 축을 결정합니다.
- **직관:** 여러 관측값을 중심 위치 하나로 요약합니다.
- **shape 확인:** `keepdim=False`이면 축약한 축이 제거되고 참이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.mean](https://docs.pytorch.org/docs/stable/generated/torch.mean.html)

</details>

<details>
<summary><code>tensor.std(dim=None, correction=1, keepdim=False)</code></summary>

#### `tensor.std(dim=None, correction=1, keepdim=False)`

- **역할:** 지정 축의 표준편차를 계산합니다.
- **입력·반환:** 텐서와 축·보정값을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 표현의 퍼짐, 데이터 scale, 학습 안정성을 진단합니다.
- **주의점:** 표본 수와 `correction`에 따라 작은 batch에서는 NaN이 될 수 있습니다.

##### 관련 수식과 코드 연결

$$
s=\sqrt{\frac{1}{N-\delta}\sum_{i=1}^{N}(x_i-\bar x)^2}
$$

- **기호:** $\delta$는 correction이며 NumPy 기본은 0, PyTorch `std` 기본은 1입니다.
- **수식 → 코드:** API의 `correction` 또는 `ddof`가 분모를 $N$과 $N-1$ 중 무엇으로 할지 정합니다.
- **직관:** 평균 주변에 값이 얼마나 퍼져 있는지를 원래 단위로 나타냅니다.
- **shape 확인:** 평균과 마찬가지로 지정한 축이 제거되거나 크기 1로 유지됩니다.
- **공식 문서:** [Tensor.std](https://docs.pytorch.org/docs/stable/generated/torch.std.html)

</details>

<details>
<summary><code>torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)</code></summary>

#### `torch.allclose(input, other, rtol=1e-5, atol=1e-8, equal_nan=False)`

- **역할:** 두 텐서의 모든 원소가 허용 오차 안에서 가까운지 검사합니다.
- **입력·반환:** 두 텐서와 오차 설정을 받고 Python `bool` 하나를 반환합니다.
- **이 셀에서 쓰는 이유:** 장치나 구현을 바꾼 뒤 수치 결과가 사실상 같은지 확인합니다.
- **주의점:** 정확한 동일성 검사가 아니며 작은 값에서는 `atol`을 직접 정해야 합니다.

##### 관련 수식과 코드 연결

$$
\lvert a_i-b_i\rvert \le \mathrm{atol}+\mathrm{rtol}\,\lvert b_i\rvert
$$

- **기호:** $a_i$는 계산값, $b_i$는 기준값, `atol`은 절대 오차, `rtol`은 상대 오차 허용치입니다.
- **수식 → 코드:** `isclose`는 이 부등식을 원소마다 계산하고, `allclose`는 모든 결과에 논리 AND를 적용합니다.
- **직관:** 완전히 같은 비트인지가 아니라 실험 목적에 충분히 가까운지를 묻는 검사입니다.
- **shape 확인:** 두 입력은 broadcasting 가능해야 합니다. `isclose`는 broadcast shape, `allclose`는 scalar `bool`을 반환합니다.
- **공식 문서:** [torch.allclose](https://docs.pytorch.org/docs/stable/generated/torch.allclose.html)

</details>

<details>
<summary><code>torch.zeros(*size, dtype=None, device=None)</code></summary>

#### `torch.zeros(*size, dtype=None, device=None)`

- **역할:** 모든 원소가 0인 텐서를 만듭니다.
- **입력·반환:** shape·dtype·device를 받고 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 초기 상태·mask·accumulator를 명시적으로 준비합니다.
- **주의점:** dtype과 device를 생략하면 주변 텐서와 달라질 수 있습니다.
- **공식 문서:** [torch.zeros](https://docs.pytorch.org/docs/stable/generated/torch.zeros.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# TODO: 직접 다시 타이핑해 보세요 — feature별 표준화
features = torch.tensor([[1.0, 10.0, 100.0], [3.0, 14.0, 120.0], [5.0, 18.0, 140.0]])
mean = features.mean(dim=0, keepdim=True)
std = features.std(dim=0, keepdim=True, unbiased=False).clamp_min(1e-6)
standardized = (features - mean) / std

print("mean:", standardized.mean(dim=0))
print("std: ", standardized.std(dim=0, unbiased=False))
assert torch.allclose(standardized.mean(0), torch.zeros(3), atol=1e-6)

# 실험: dim=0을 dim=1로 바꾸면 무엇을 표준화하게 될까요?

## 3. Device 관리

모델과 입력은 같은 장치에 있어야 합니다. 이 저장소는 `AI_LAB_DEVICE=auto`에서 CUDA → MPS → CPU 순으로 선택합니다. 재현성 비교가 필요하면 실행 전에 `AI_LAB_DEVICE=cpu`로 강제할 수 있습니다.

실무 팁: checkpoint는 `map_location="cpu"`로 먼저 읽고, 모델을 원하는 장치로 옮기면 이식성이 좋아집니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Linear,Module.parameters -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Linear(in_features, out_features, bias=True)</code></summary>

#### `nn.Linear(in_features, out_features, bias=True)`

- **역할:** 완전연결 affine 변환을 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
y=xW^{\mathsf T}+b
$$

- **기호:** $x$는 입력, $W$는 weight, $b$는 bias, $y$는 출력입니다.
- **수식 → 코드:** `in_features`와 `out_features`가 각각 $W$의 두 축을 결정합니다.
- **직관:** 각 출력 feature는 모든 입력 feature의 학습 가능한 가중합입니다.
- **shape 확인:** `[..., D_in] -> [..., D_out]`, weight는 `[D_out, D_in]`입니다.
- **공식 문서:** [nn.Linear](https://docs.pytorch.org/docs/stable/generated/torch.nn.Linear.html)

</details>

<details>
<summary><code>model.parameters(recurse=True)</code></summary>

#### `model.parameters(recurse=True)`

- **역할:** 모듈에 등록된 학습 파라미터 iterator를 반환합니다.
- **입력·반환:** 하위 모듈 포함 여부를 받고 `Parameter` iterator를 반환합니다.
- **이 셀에서 쓰는 이유:** optimizer가 갱신할 weight와 bias를 전달합니다.
- **주의점:** iterator는 한 번 소비되며 buffer와 일반 tensor attribute는 포함하지 않습니다.
- **공식 문서:** [Module.parameters](https://docs.pytorch.org/docs/stable/generated/torch.nn.Module.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
device = ACCELERATOR.device

model = nn.Linear(4, 2).to(device)
inputs = torch.randn(3, 4).to(device)
outputs = model(inputs)

print("parameter device:", next(model.parameters()).device)
print("input/output:", inputs.device, outputs.device)
assert next(model.parameters()).device == inputs.device == outputs.device

## 4. Autograd: 계산 그래프를 역방향으로 따라가기

`requires_grad=True`인 leaf tensor에서 시작한 연산은 계산 그래프를 만듭니다. 최종 scalar loss에 `backward()`를 호출하면 chain rule로 각 leaf의 `.grad`가 채워집니다.

$$y = x^2 + 3x,\quad \frac{dy}{dx}=2x+3$$

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.backward,Tensor.item -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>loss.backward()</code></summary>

#### `loss.backward()`

- **역할:** 스칼라 loss에서 계산 그래프를 거슬러 gradient를 계산합니다.
- **입력·반환:** 스칼라 텐서에서는 보통 인자 없이 호출하며 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** optimizer update에 필요한 각 파라미터의 `.grad`를 채웁니다.
- **주의점:** 같은 그래프를 다시 역전파하려면 별도 보존 설정이 필요합니다.

##### 관련 수식과 코드 연결

$$
\frac{\partial L}{\partial x}=\frac{\partial L}{\partial y}\frac{\partial y}{\partial x},\qquad x.\mathrm{grad}\leftarrow x.\mathrm{grad}+\nabla_xL
$$

- **기호:** $L$은 scalar loss이고 $x$는 계산 그래프의 leaf parameter입니다.
- **수식 → 코드:** `backward()`는 연쇄법칙으로 gradient를 계산해 각 parameter의 `.grad`에 누적합니다.
- **직관:** 파라미터를 직접 바꾸는 호출이 아니라 update에 필요한 미분값을 준비하는 호출입니다.
- **shape 확인:** 각 `.grad`는 대응 parameter와 같은 shape이며 scalar가 아닌 출력에는 seed gradient가 필요합니다.
- **공식 문서:** [Tensor.backward](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.backward.html)

</details>

<details>
<summary><code>tensor.item()</code></summary>

#### `tensor.item()`

- **역할:** 원소 하나짜리 텐서에서 Python 숫자를 꺼냅니다.
- **입력·반환:** 단일 원소 텐서를 받고 Python 숫자를 반환합니다.
- **이 셀에서 쓰는 이유:** loss나 metric을 출력·기록 가능한 값으로 바꿉니다.
- **주의점:** 여러 원소 텐서에는 쓸 수 없고 GPU에서는 동기화를 일으킬 수 있습니다.
- **공식 문서:** [Tensor.item](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.item.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
x = torch.tensor(2.0, requires_grad=True)
y = x**2 + 3 * x
y.backward()

print("y:", y.item())
print("autograd dy/dx:", x.grad.item())
print("analytic dy/dx:", 2 * x.item() + 3)
assert torch.allclose(x.grad, torch.tensor(7.0))

In [ ]:
features = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
weights = torch.randn(2, 1, requires_grad=True)  # leaf parameter
predictions = features @ weights  # non-leaf tensor
loss = predictions.square().mean()  # scalar

print("weights: leaf=", weights.is_leaf, "grad_fn=", weights.grad_fn)
print("predictions: leaf=", predictions.is_leaf, "grad_fn=", predictions.grad_fn)
print("loss grad_fn:", loss.grad_fn)
loss.backward()
print("weights.grad shape:", weights.grad.shape)
assert weights.grad.shape == weights.shape

In [ ]:
# gradient는 기본적으로 누적됩니다. 미니배치마다 초기화해야 합니다.
w = torch.tensor(2.0, requires_grad=True)
(w**2).backward()
first = w.grad.item()
(w**2).backward()
accumulated = w.grad.item()
print("first:", first, "after second backward:", accumulated)
assert accumulated == 2 * first

w.grad = None  # optimizer.zero_grad(set_to_none=True)와 같은 의도
(w**2).backward()
print("after reset:", w.grad.item())

### `detach`, `no_grad`, `inference_mode`

- `detach()`: 특정 tensor를 현재 그래프에서 분리
- `torch.no_grad()`: 블록 안의 gradient 기록 중지
- `torch.inference_mode()`: 추론 전용으로 더 강한 최적화 적용

평가/배포 추론에는 `model.eval()`과 `inference_mode()`를 함께 사용합니다. `eval()`은 dropout·batch norm 동작을 바꾸지만 gradient 기록을 끄지는 않습니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=Tensor.detach,torch.no_grad,torch.inference_mode -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 0개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>tensor.detach()</code></summary>

#### `tensor.detach()`

- **역할:** 같은 저장공간을 공유하면서 현재 계산 그래프에서 분리한 텐서를 만듭니다.
- **입력·반환:** 텐서를 받아 gradient를 추적하지 않는 텐서 view를 반환합니다.
- **이 셀에서 쓰는 이유:** 로그·teacher target·NumPy 변환이 역전파 경로를 만들지 않게 합니다.
- **주의점:** 원본과 저장공간을 공유하므로 in-place 변경은 양쪽에 영향을 줄 수 있습니다.
- **공식 문서:** [Tensor.detach](https://docs.pytorch.org/docs/stable/generated/torch.Tensor.detach.html)

</details>

<details>
<summary><code>torch.no_grad()</code></summary>

#### `torch.no_grad()`

- **역할:** 블록 안에서 autograd 기록을 끄는 context manager입니다.
- **입력·반환:** 보통 `with`에 사용하며 블록의 텐서는 기본적으로 gradient를 기록하지 않습니다.
- **이 셀에서 쓰는 이유:** 평가나 고정 teacher 추론의 메모리와 연산 부담을 줄입니다.
- **주의점:** `model.eval()`과 역할이 다르므로 평가 모드가 필요하면 둘 다 사용합니다.
- **공식 문서:** [torch.no_grad](https://docs.pytorch.org/docs/stable/generated/torch.no_grad.html)

</details>

<details>
<summary><code>torch.inference_mode(mode=True)</code></summary>

#### `torch.inference_mode(mode=True)`

- **역할:** 추론 전용으로 autograd 관련 추적을 더 강하게 비활성화합니다.
- **입력·반환:** 불리언 모드를 받고 context manager 또는 decorator로 동작합니다.
- **이 셀에서 쓰는 이유:** 순수 평가 구간의 오버헤드를 줄입니다.
- **주의점:** 이 모드에서 만든 텐서는 이후 gradient 계산에 쓰기 어려울 수 있습니다.
- **공식 문서:** [torch.inference_mode](https://docs.pytorch.org/docs/stable/generated/torch.inference_mode.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
source = torch.tensor([1.0, 2.0], requires_grad=True)
tracked = source * 3
detached = tracked.detach()

with torch.no_grad():
    no_grad_value = source * 3
with torch.inference_mode():
    inference_value = source * 3

print("tracked requires_grad:  ", tracked.requires_grad)
print("detached requires_grad: ", detached.requires_grad)
print("no_grad requires_grad:  ", no_grad_value.requires_grad)
print("inference requires_grad:", inference_value.requires_grad)

## 5. 가장 작은 학습 루프

아래에서는 $y=2x+1$을 따르는 데이터에서 두 parameter `w`, `b`를 찾습니다.

학습 루프의 고정 순서는 `zero_grad → forward → loss → backward → step`입니다.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.optim.SGD,Optimizer.zero_grad,Optimizer.step -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 3개 · 수식 2개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.optim.SGD(params, lr, momentum=0, weight_decay=0, ...)</code></summary>

#### `torch.optim.SGD(params, lr, momentum=0, weight_decay=0, ...)`

- **역할:** gradient 방향으로 파라미터를 이동하는 SGD optimizer를 만듭니다.
- **입력·반환:** 파라미터와 학습률·momentum 등을 받고 optimizer 객체를 반환합니다.
- **이 셀에서 쓰는 이유:** 고전 CNN과 최적화 기초 실험의 update 규칙을 명시합니다.
- **주의점:** momentum·weight decay·Nesterov 설정에 따라 실제 update 식이 달라집니다.

##### 관련 수식과 코드 연결

$$
g_t=\nabla_\theta L_t,\qquad v_t=\mu v_{t-1}+g_t,\qquad\theta_{t+1}=\theta_t-\eta v_t
$$

- **기호:** $\eta$는 학습률, $\mu$는 momentum 계수입니다. $\mu=0$이면 기본 SGD입니다.
- **수식 → 코드:** `momentum`이 0이면 현재 gradient만, 양수이면 이전 update 방향도 함께 사용합니다.
- **직관:** 손실이 가장 빠르게 증가하는 gradient의 반대 방향으로 parameter를 이동합니다.
- **shape 확인:** 각 momentum buffer는 대응 parameter와 같은 shape입니다.
- **공식 문서:** [torch.optim.SGD](https://docs.pytorch.org/docs/stable/generated/torch.optim.SGD.html)

</details>

<details>
<summary><code>optimizer.zero_grad(set_to_none=True)</code></summary>

#### `optimizer.zero_grad(set_to_none=True)`

- **역할:** 파라미터에 누적된 gradient를 비웁니다.
- **입력·반환:** 초기화 방식을 받고 반환값은 없습니다.
- **이 셀에서 쓰는 이유:** 현재 mini-batch의 gradient만으로 update하도록 학습 단계를 시작합니다.
- **주의점:** 호출하지 않으면 PyTorch gradient가 기본적으로 누적됩니다.
- **공식 문서:** [Optimizer.zero_grad](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.zero_grad.html)

</details>

<details>
<summary><code>optimizer.step(closure=None)</code></summary>

#### `optimizer.step(closure=None)`

- **역할:** 현재 gradient와 optimizer 규칙으로 파라미터를 갱신합니다.
- **입력·반환:** 선택적 closure를 받고 optimizer에 따라 loss 또는 `None`을 반환합니다.
- **이 셀에서 쓰는 이유:** 역전파로 계산한 gradient를 실제 학습 update에 반영합니다.
- **주의점:** 일반적으로 `zero_grad`, forward, `backward` 다음에 호출합니다.

##### 관련 수식과 코드 연결

$$
\theta_{t+1}=\mathcal{U}(\theta_t,\nabla_\theta L_t,s_t;\eta)
$$

- **기호:** $\mathcal U$는 optimizer update, $s_t$는 momentum·moment 상태입니다.
- **수식 → 코드:** `step()`은 이미 `.grad`에 저장된 값을 읽어 선택한 optimizer 규칙을 적용합니다.
- **직관:** gradient 계산과 parameter 갱신을 분리하면 학습 lifecycle을 정확히 추적할 수 있습니다.
- **shape 확인:** parameter shape은 유지되고 값과 optimizer state만 바뀝니다.
- **공식 문서:** [Optimizer.step](https://docs.pytorch.org/docs/stable/generated/torch.optim.Optimizer.step.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
torch.manual_seed(7)
x_train = torch.linspace(-1, 1, 32).unsqueeze(1)
y_train = 2 * x_train + 1 + 0.05 * torch.randn_like(x_train)

linear = nn.Linear(1, 1)
optimizer = torch.optim.SGD(linear.parameters(), lr=0.15)

for step in range(101):
    optimizer.zero_grad(set_to_none=True)
    prediction = linear(x_train)
    loss = ((prediction - y_train) ** 2).mean()
    loss.backward()
    optimizer.step()
    if step % 25 == 0:
        print(f"step={step:3d} loss={loss.item():.6f}")

learned_w = linear.weight.item()
learned_b = linear.bias.item()
print(f"learned: y = {learned_w:.3f}x + {learned_b:.3f}")
assert abs(learned_w - 2.0) < 0.1 and abs(learned_b - 1.0) < 0.1

In [ ]:
# 수치 미분으로 autograd 검산하기
def f(value: torch.Tensor) -> torch.Tensor:
    return (value**3 + 2 * value).sum()


point = torch.tensor([1.5], requires_grad=True)
f(point).backward()
automatic = point.grad.detach().clone()

epsilon = 1e-4
center = point.detach()
numerical = (f(center + epsilon) - f(center - epsilon)) / (2 * epsilon)
print("autograd:", automatic.item(), "numerical:", numerical.item())
assert torch.allclose(automatic, numerical, rtol=1e-3, atol=1e-3)

## 6. Gradient를 관찰하는 디버깅 도구

hook은 backward 중 특정 tensor나 parameter의 gradient를 관찰합니다. 큰 모델에서는 모든 값을 출력하지 말고 norm, min/max, 유한성만 기록하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.Sequential,nn.ReLU,torch.linalg.norm,F.cross_entropy -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 4개 · 수식 3개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.Sequential(*modules)</code></summary>

#### `nn.Sequential(*modules)`

- **역할:** 모듈을 순서대로 연결합니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.
- **공식 문서:** [nn.Sequential](https://docs.pytorch.org/docs/stable/generated/torch.nn.Sequential.html)

</details>

<details>
<summary><code>nn.ReLU(inplace=False)</code></summary>

#### `nn.ReLU(inplace=False)`

- **역할:** 음수 값을 0으로 만드는 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{ReLU}(x)=\max(0,x)
$$

- **기호:** $x$는 선형 변환 뒤 activation입니다.
- **수식 → 코드:** 코드가 음수 원소를 0으로 치환하고 양수는 그대로 통과시킵니다.
- **직관:** 단순하지만 여러 선형 층을 비선형 함수로 조합할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.ReLU](https://docs.pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

</details>

<details>
<summary><code>torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)</code></summary>

#### `torch.linalg.vector_norm(x, ord=2, dim=None, keepdim=False)`

- **역할:** 벡터의 크기인 norm을 계산합니다.
- **입력·반환:** 텐서·차수·축을 받고 norm 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 거리, gradient 크기, 정규화 분모를 계산합니다.
- **주의점:** `dim`과 norm 차수에 따라 의미와 반환 shape이 달라집니다.

##### 관련 수식과 코드 연결

$$
\lVert x\rVert_p=\left(\sum_i\lvert x_i\rvert^p\right)^{1/p},\qquad \lVert x\rVert_2=\sqrt{\sum_i x_i^2}
$$

- **기호:** $p$는 norm 차수이며 $p=2$가 유클리드 길이입니다.
- **수식 → 코드:** `ord`와 `dim`·`axis`가 어떤 원소 묶음의 크기를 계산할지 정합니다.
- **직관:** 여러 성분을 하나의 거리·크기 값으로 요약합니다.
- **shape 확인:** 지정한 축은 결과에서 제거되며 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [torch.linalg.norm](https://docs.pytorch.org/docs/stable/generated/torch.linalg.vector_norm.html)

</details>

<details>
<summary><code>F.cross_entropy(input, target, ...)</code></summary>

#### `F.cross_entropy(input, target, ...)`

- **역할:** logit에 log-softmax와 음의 로그우도를 결합합니다.
- **입력·반환:** 예측·target과 설정을 받고 축약된 loss 또는 같은 shape의 loss를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 objective를 수치화해 역전파 시작점으로 사용합니다.
- **주의점:** 입력 형식, target dtype·shape, `reduction` 의미를 반드시 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\mathcal{L}_{CE}=-\frac{1}{N}\sum_{n=1}^{N}\log\frac{e^{z_{n,y_n}}}{\sum_{c=1}^{C}e^{z_{n,c}}}
$$

- **기호:** $z_{n,c}$는 class logit, $y_n$은 정답 class ID, $C$는 class 수입니다.
- **수식 → 코드:** 함수가 내부에서 `log_softmax`와 정답 class의 음의 로그우도를 결합합니다.
- **직관:** 정답 class 확률이 낮을수록 큰 벌점을 줍니다.
- **shape 확인:** 일반적으로 logits `[N,C,...]`, class-index target `[N,...]`을 사용합니다.
- **공식 문서:** [F.cross_entropy](https://docs.pytorch.org/docs/stable/generated/torch.nn.functional.cross_entropy.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
probe = nn.Sequential(nn.Linear(4, 8), nn.ReLU(), nn.Linear(8, 2))
gradient_log = {}

handles = []
for name, parameter in probe.named_parameters():

    def save_norm(grad, parameter_name=name):
        gradient_log[parameter_name] = float(grad.norm())

    handles.append(parameter.register_hook(save_norm))

batch = torch.randn(5, 4)
target = torch.tensor([0, 1, 0, 1, 1])
loss = nn.functional.cross_entropy(probe(batch), target)
loss.backward()

print("loss:", loss.item())
print("gradient norms:", gradient_log)
assert gradient_log and all(value >= 0 for value in gradient_log.values())
for handle in handles:
    handle.remove()

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.log -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.log(input)</code></summary>

#### `torch.log(input)`

- **역할:** 각 원소에 자연로그를 적용합니다.
- **입력·반환:** 양수 텐서를 받고 같은 shape의 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 곱을 합으로 바꾸고 log-likelihood를 계산합니다.
- **주의점:** 0은 음의 무한대, 음수는 NaN이므로 작은 epsilon이나 clamp가 필요합니다.

##### 관련 수식과 코드 연결

$$
y_i=\log x_i,\qquad x_i>0
$$

- **기호:** $\log$는 자연로그입니다.
- **수식 → 코드:** API가 곱셈 관계를 덧셈 관계로 바꾸고 작은 확률의 크기를 안정적으로 표현합니다.
- **직관:** 확률 곱을 log-probability 합으로 계산할 수 있게 합니다.
- **shape 확인:** 입력과 출력 shape은 같으며 입력은 양수여야 합니다.
- **공식 문서:** [torch.log](https://docs.pytorch.org/docs/stable/generated/torch.log.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
# NaN/Inf가 생기는 지점을 빨리 찾는 작은 계약 함수
def check_tensor(
    name: str, tensor: torch.Tensor, expected_ndim: int | None = None
) -> None:
    if expected_ndim is not None:
        assert tensor.ndim == expected_ndim, (
            f"{name}: expected ndim={expected_ndim}, got {tensor.shape}"
        )
    assert torch.isfinite(tensor).all(), f"{name}: NaN 또는 Inf 발견"


healthy = torch.log(torch.tensor([1.0, 2.0, 3.0]))
check_tensor("healthy", healthy, expected_ndim=1)

unhealthy = torch.tensor([1.0, float("nan")])
try:
    check_tensor("unhealthy", unhealthy)
except AssertionError as error:
    print("검사가 문제를 잡았습니다:", error)

<!-- api-explanations:start -->
<!-- api-explanations:keys=nn.GELU -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 1개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>nn.GELU(approximate=&#x27;none&#x27;)</code></summary>

#### `nn.GELU(approximate='none')`

- **역할:** 부드러운 GELU 활성함수를 만듭니다.
- **입력·반환:** 구조 hyperparameter를 받고 호출 가능한 `nn.Module` 또는 파라미터를 반환합니다.
- **이 셀에서 쓰는 이유:** 논문의 학습 가능한 변환을 명시적인 구성 요소로 조립합니다.
- **주의점:** 입·출력 shape, bias 여부, 등록 상태가 논문 수식과 같은지 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
\operatorname{GELU}(x)=x\Phi(x)\approx\frac{x}{2}\left[1+\tanh\!\left(\sqrt{\frac{2}{\pi}}(x+0.044715x^3)\right)\right]
$$

- **기호:** $\Phi(x)$는 표준정규분포의 누적분포함수입니다.
- **수식 → 코드:** `approximate` 설정에 따라 정확한 CDF 기반 값 또는 tanh 근사를 계산합니다.
- **직관:** 입력 크기에 비례해 부드럽게 gate하여 ReLU보다 경계가 매끄럽습니다.
- **shape 확인:** 입력과 출력 shape은 같습니다.
- **공식 문서:** [nn.GELU](https://docs.pytorch.org/docs/stable/generated/torch.nn.GELU.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
class ShapeCheckedMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dim: int, output_dim: int):
        super().__init__()
        self.input_dim = input_dim
        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.ndim != 2 or x.shape[-1] != self.input_dim:
            raise ValueError(
                f"expected [batch, {self.input_dim}], got {tuple(x.shape)}"
            )
        logits = self.network(x)
        assert logits.shape[0] == x.shape[0]
        return logits


checked_model = ShapeCheckedMLP(4, 8, 3)
print("valid output:", checked_model(torch.randn(2, 4)).shape)
try:
    checked_model(torch.randn(2, 5))
except ValueError as error:
    print("명확한 오류:", error)

## 7. 따라 치기 TODO

아래 셀은 실행 가능한 정답 형태입니다. 먼저 주석을 읽고 **새 셀에 보지 않고 다시 작성**해 보세요.

1. `[B,T,D]` 텐서에서 time 평균을 내 `[B,D]`를 만드세요.
2. padding mask `[B,T]`를 적용한 masked mean을 구현하세요.
3. 두 층 MLP의 모든 parameter에 gradient가 들어왔는지 검사하세요.

<!-- api-explanations:start -->
<!-- api-explanations:keys=torch.bool,Tensor.sum -->
<details>
<summary><strong>이 셀에서 처음 만나는 함수·API·수식 해설 (API 2개 · 수식 1개)</strong></summary>

> 아래 내용은 라이브러리 API의 의미를 설명합니다. 실습 정답 로직은 포함하지 않습니다.
> 수식은 별도 표기가 없으면 대표적인 기본 설정을 나타냅니다. `dim`, `reduction`, 가중치 같은 실제 인자에 따라 세부 형태가 달라질 수 있습니다.

<details>
<summary><code>torch.bool</code></summary>

#### `torch.bool`

- **역할:** 불리언 dtype
- **입력·반환:** 값이 아니라 텐서 원소의 저장 형식으로 사용됩니다.
- **이 셀에서 쓰는 이유:** mask와 조건 결과를 표현합니다.
- **주의점:** 연산마다 허용 dtype이 다르고 변환 시 정밀도나 메모리가 달라집니다.
- **공식 문서:** [torch.bool](https://docs.pytorch.org/docs/stable/tensor_attributes.html#torch.dtype)

</details>

<details>
<summary><code>tensor.sum(dim=None, keepdim=False, dtype=None)</code></summary>

#### `tensor.sum(dim=None, keepdim=False, dtype=None)`

- **역할:** 지정 축의 원소 합을 계산합니다.
- **입력·반환:** 텐서와 축 설정을 받고 축약된 텐서를 반환합니다.
- **이 셀에서 쓰는 이유:** 확률 질량, mask 개수, message aggregation을 계산합니다.
- **주의점:** 정수 overflow와 축약 뒤 shape 변화를 확인해야 합니다.

##### 관련 수식과 코드 연결

$$
S=\sum_{i=1}^{N}x_i
$$

- **기호:** $N$은 선택 축의 원소 수입니다.
- **수식 → 코드:** `dim` 또는 `axis`의 모든 원소를 더해 축약합니다.
- **직관:** count, 확률 질량, 이웃 message처럼 누적량을 계산합니다.
- **shape 확인:** 선택한 축은 제거되거나 `keepdim=True`이면 크기 1로 남습니다.
- **공식 문서:** [Tensor.sum](https://docs.pytorch.org/docs/stable/generated/torch.sum.html)

</details>

</details>
<!-- api-explanations:end -->

In [ ]:
sequence = torch.randn(2, 4, 3)  # [B, T, D]
attention_mask = torch.tensor([[1, 1, 1, 1], [1, 1, 0, 0]], dtype=torch.bool)

# TODO 1: time 축 평균
plain_mean = sequence.mean(dim=1)

# TODO 2: padding을 제외한 masked mean
weights = attention_mask.unsqueeze(-1).to(sequence.dtype)
masked_mean = (sequence * weights).sum(dim=1) / weights.sum(dim=1).clamp_min(1.0)

# TODO 3: gradient 연결 검사
exercise_model = nn.Sequential(nn.Linear(3, 5), nn.Tanh(), nn.Linear(5, 2))
exercise_model(masked_mean).sum().backward()
missing_grad = [name for name, p in exercise_model.named_parameters() if p.grad is None]

print("plain_mean:", plain_mean.shape, "masked_mean:", masked_mean.shape)
print("gradient가 없는 parameter:", missing_grad)
assert masked_mean.shape == (2, 3)
assert missing_grad == []

## 마무리 체크리스트

모델이 예상대로 학습되지 않을 때 다음 순서로 확인하세요.

1. 입력 한 배치의 `shape/dtype/device/range`를 출력한다.
2. 레이블 범위가 `[0, num_classes)`인지 확인한다.
3. forward 중간 shape에 `assert`를 둔다.
4. loss가 finite인지 확인한다.
5. `zero_grad → backward → step` 순서를 확인한다.
6. parameter별 `grad is None`, gradient norm을 확인한다.
7. 작은 배치 하나를 과적합할 수 있는지 시험한다.

다음 노트북에서는 이 원칙을 실제 `Dataset → DataLoader → 텍스트 분류기 → checkpoint` 파이프라인에 적용합니다.